In [2]:
import os
from pathlib import Path
import cv2
import numpy as np
import pandas as pd

BASE_DATA_DIR = Path(r"D:\Semesters\2026-1(6th semester)\AI 개론\Team Project\animal_face_images")
TEST_DATA_DIR = Path(r"D:\Semesters\2026-1(6th semester)\AI 개론\Team Project\test_images")

IMG_SIZE = 64

LABEL_MAP = {
    '강아지상': 0,
    '고양이상': 1,
    '토끼상': 2,
    '곰상': 3, #곰상이랑 토끼상은 빠른 시일내에 사진 넣어 놓겠습니다!
    '여우상': 4 
}

face_cascade = cv2.CascadeClassifier(cv2.data.haarcascades + 'haarcascade_frontalface_default.xml')

print("✅ [설정 완료] 영웅 님 컴퓨터 경로 세팅이 정상적으로 끝났습니다.")
print(f" - 학습 데이터 경로: {BASE_DATA_DIR}")
print(f" - 판별용 데이터 경로: {TEST_DATA_DIR}\n")

flattened_data = []
labels = []

print("=== [1단계] 학습 데이터셋 전처리 가동 ===")
for folder_name, label_value in LABEL_MAP.items():
    folder_path = BASE_DATA_DIR / folder_name
    
    if not folder_path.exists():
        print(f"⚠️ 경고: '{folder_name}' 폴더를 찾을 수 없어 건너뜁니다.")
        continue
        
    print(f"▶ '{folder_name}' 처리 중... (Label: {label_value})")
    
    for filename in os.listdir(folder_path):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
            img_path = folder_path / filename
            
            img_array = np.fromfile(str(img_path), np.uint8)
            img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
            
            if img is None: continue
            
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
            
            if len(faces) > 0:
                x, y, w, h = faces[0]
                cropped_face = gray[y:y+h, x:x+w]
            else:
                cropped_face = gray
                
            resized_face = cv2.resize(cropped_face, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
            flattened_data.append(resized_face.flatten())
            labels.append(label_value)

if len(flattened_data) > 0:
    pixel_columns = [f"pixel_{i}" for i in range(IMG_SIZE * IMG_SIZE)]
    df_train = pd.DataFrame(flattened_data, columns=pixel_columns)
    df_train['Label'] = labels
    
    df_train.to_csv("./animal_face_dataset.csv", index=False, encoding='utf-8-sig')
    print(f"💾 학습 데이터 구축 완료 -> 'animal_face_dataset.csv' 저장됨! (크기: {df_train.shape})\n")

test_flattened_data = []
test_file_names = []

print("=== [2단계] 판별용(test_images) 데이터셋 전처리 가동 ===")
if not TEST_DATA_DIR.exists():
    print(f"💡 안내: '{TEST_DATA_DIR}' 폴더가 없습니다. 폴더를 생성해 주세요!")
else:
    for filename in os.listdir(TEST_DATA_DIR):
        if filename.lower().endswith(('.png', '.jpg', '.jpeg', '.webp')):
            img_path = TEST_DATA_DIR / filename
            
            img_array = np.fromfile(str(img_path), np.uint8)
            img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
            
            if img is None: continue
                
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            faces = face_cascade.detectMultiScale(gray, scaleFactor=1.1, minNeighbors=5, minSize=(30, 30))
            
            if len(faces) > 0:
                x, y, w, h = faces[0]
                cropped_face = gray[y:y+h, x:x+w]
            else:
                cropped_face = gray
                
            resized_face = cv2.resize(cropped_face, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
            test_flattened_data.append(resized_face.flatten())
            test_file_names.append(filename)

    if len(test_flattened_data) > 0:
        pixel_columns = [f"pixel_{i}" for i in range(IMG_SIZE * IMG_SIZE)]
        df_test = pd.DataFrame(test_flattened_data, columns=pixel_columns)
        df_test['Origin_Filename'] = test_file_names
        
        df_test.to_csv("./unseen_test_dataset.csv", index=False, encoding='utf-8-sig')
        print(f"💾 판별용 데이터 구축 완료 -> 'unseen_test_dataset.csv' 저장됨! (개수: {df_test.shape[0]}개)")
    else:
        print("💡 안내: test_images 폴더에 넣은 사진이 없거나 읽을 수 없습니다.")

        

✅ [설정 완료] 영웅 님 컴퓨터 경로 세팅이 정상적으로 끝났습니다.
 - 학습 데이터 경로: D:\Semesters\2026-1(6th semester)\AI 개론\Team Project\animal_face_images
 - 판별용 데이터 경로: D:\Semesters\2026-1(6th semester)\AI 개론\Team Project\test_images

=== [1단계] 학습 데이터셋 전처리 가동 ===
⚠️ 경고: '강아지상' 폴더를 찾을 수 없어 건너뜁니다.
⚠️ 경고: '고양이상' 폴더를 찾을 수 없어 건너뜁니다.
⚠️ 경고: '토끼상' 폴더를 찾을 수 없어 건너뜁니다.
⚠️ 경고: '곰상' 폴더를 찾을 수 없어 건너뜁니다.
⚠️ 경고: '여우상' 폴더를 찾을 수 없어 건너뜁니다.
=== [2단계] 판별용(test_images) 데이터셋 전처리 가동 ===
💡 안내: 'D:\Semesters\2026-1(6th semester)\AI 개론\Team Project\test_images' 폴더가 없습니다. 폴더를 생성해 주세요!


In [3]:
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

IMG_SIZE = 64

LABEL_NAME = {
    0: "강아지상",
    1: "고양이상",
    2: "토끼상",
    3: "곰상",
    4: "여우상"
}

df = pd.read_csv("./animal_face_dataset.csv")

pixel_cols = [col for col in df.columns if col.startswith("pixel_")]

X = df[pixel_cols].values / 255.0
y = df["Label"].values

X_train, X_val, y_train, y_val = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

templates = {}

for label in LABEL_NAME.keys():
    templates[label] = X_train[y_train == label].mean(axis=0)

print("동물상별 평균 얼굴 템플릿 생성 완료")

for label in LABEL_NAME.keys():
    plt.figure(figsize=(3, 3))

    plt.imshow(
        templates[label].reshape(
            IMG_SIZE,
            IMG_SIZE
        ),
        cmap="gray"
    )

    plt.title(
        LABEL_NAME[label]
    )

    plt.axis("off")
    plt.show()

def predict_template_matching(image):

    distances = {}

    for label in LABEL_NAME.keys():

        distance = np.linalg.norm(
            image - templates[label]
        )

        distances[label] = distance

    pred_label = min(
        distances,
        key=distances.get
    )

    return pred_label, distances

y_pred = []

for image in X_val:

    pred_label, distances = predict_template_matching(
        image
    )

    y_pred.append(pred_label)

accuracy = accuracy_score(
    y_val,
    y_pred
)

print()
print("템플릿 매칭 1차 정확도")

print(
    f"Accuracy : {accuracy:.4f}"
)

print()
print("Confusion Matrix")

print(
    confusion_matrix(
        y_val,
        y_pred,
        labels=list(LABEL_NAME.keys())
    )
)

print()
print("Classification Report")

print(
    classification_report(
        y_val,
        y_pred,
        labels=list(LABEL_NAME.keys()),
        target_names=list(
            LABEL_NAME.values()
        )
    )
)

FileNotFoundError: [Errno 2] No such file or directory: './animal_face_dataset.csv'

In [ ]:
df_test = pd.read_csv("./unseen_test_dataset.csv")

X_test = df_test[pixel_cols].values / 255.0
file_names = df_test["Origin_Filename"].values

results = []

for filename, x in zip(file_names, X_test):
    pred_label, distances = predict_template_matching(x)

    result = {
        "filename": filename,
        "pred_label": pred_label,
        "pred_animal": LABEL_NAME.get(pred_label, str(pred_label))
    }

    for label, dist in distances.items():
        result[f"distance_{LABEL_NAME.get(label, label)}"] = dist

    results.append(result)

df_result = pd.DataFrame(results)
df_result.to_csv("./template_matching_result.csv", index=False, encoding="utf-8-sig")

df_result.head()